# Supplementary Figure: Multi-depth Receptive Fields vs. Single-depth Receptive Fields

This notebook generates the publication-ready supplementary figure comparing receptive fields measured under multi-depth stimulus conditions with standard single-depth receptive fields in V1:
- **A, B, C**: Example neurons representing Near, Mid, and Far depth tuning, showing their 1D depth tuning curves and side-by-side 2D spatial receptive fields (Single depth vs Multi depth) across 8 virtual depths.
- **D**: Population distribution of receptive field correlations between single-depth and multi-depth conditions (Contralateral vs. Ipsilateral control).
- **E**: Preferred virtual depth consistency: Multi-depth RF peak depth vs. Single-depth RF peak depth (cm).


In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import numpy.core as np_core
# NumPy backward-compatibility shim for pickled data
sys.modules.setdefault("numpy._core", np_core)
sys.modules.setdefault("numpy._core.numeric", np_core.numeric)
sys.modules.setdefault("numpy._core.multiarray", np_core.multiarray)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42  # for pdfs
matplotlib.rcParams['svg.fonttype'] = 'none'  # for svgs
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

import flexiznam as flz
from cottage_analysis.analysis.spheres import rf_analysis, spheres
from cottage_analysis.plotting import depth_selectivity_plots, rf_plots, rsof_plots, plotting_utils, style
from cottage_analysis.pipelines import pipeline_utils
from v1_depth_map.revisions.revision_sessions import sessions
from v1_depth_map.paths import get_figures_roots

In [ ]:
# Set default font to Arial
arial_font_path = "/Volumes/BlackPasspo/v1_depth_map/processed/v1_manuscript_figures/fonts/arial.ttf"
if arial_font_path:
    try:
        arial_prop = fm.FontProperties(fname=arial_font_path)
        plt.rcParams["font.family"] = arial_prop.get_name()
        plt.rcParams.update({"mathtext.default": "regular"})
        fm.fontManager.addfont(arial_font_path)
    except Exception:
        pass

In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
(SAVE_ROOT / "fig_supp_multidepth_rf").mkdir(parents=True, exist_ok=True)

## 1. Load Population & Session Data

In [ ]:
# Load all multi-depth revision sessions
multi_depth_sessions = [k for k, v in sessions.items() if v == "multidepth"]
print(f"Loading data for {len(multi_depth_sessions)} multi-depth sessions: {multi_depth_sessions}")

all_sig_m, all_sig_ipsi_m, neurons_df = rf_analysis.load_sig_rf(
    flexilims_session=flexilims_session,
    session_list=multi_depth_sessions,
    n_std=6,
    filter_datasets={"annotated": True},
    use_multidepth=True,
    use_cols=None,
    sphere_presentation_mask=None,
)

ndepths = 8
# Compute RF spatial centers and peak depth for single-depth protocol
coef_single = np.stack(neurons_df["rf_coef_closedloop"].values)
azi_s, ele_s, idepth_s, _ = rf_analysis.find_rf_centers(
    neurons_df, ndepths=ndepths, frame_shape=(16, 24), is_closed_loop=1, resolution=5, coef=coef_single
)
neurons_df["rf_azi_single"] = azi_s
neurons_df["rf_ele_single"] = ele_s
neurons_df["rf_idepth_single"] = idepth_s

# Compute RF spatial centers and peak depth for multi-depth protocol
coef_multi = np.stack(neurons_df["rf_coef_closedloop_multidepth"].values)
azi_m, ele_m, idepth_m, _ = rf_analysis.find_rf_centers(
    neurons_df, ndepths=ndepths, frame_shape=(16, 24), is_closed_loop=1, resolution=5, coef=coef_multi
)
neurons_df["rf_azi_multi"] = azi_m
neurons_df["rf_ele_multi"] = ele_m
neurons_df["rf_idepth_multi"] = idepth_m

# Calculate pixel-by-pixel full-volume correlation between single-depth and multi-depth RFs
def get_rf_corr(row):
    try:
        c_single = row["rf_coef_closedloop"][:, :-1].flatten()
        c_multi = row["rf_coef_closedloop_multidepth"][:, :-1].flatten()
        valid = ~np.isnan(c_single) & ~np.isnan(c_multi)
        if valid.sum() > 10:
            return np.corrcoef(c_single[valid], c_multi[valid])[0, 1]
        return np.nan
    except Exception:
        return np.nan

def get_rf_ipsi_corr(row):
    try:
        c_single = row["rf_coef_ipsi_closedloop"][:, :-1].flatten()
        c_multi = row["rf_coef_ipsi_closedloop_multidepth"][:, :-1].flatten()
        valid = ~np.isnan(c_single) & ~np.isnan(c_multi)
        if valid.sum() > 10:
            return np.corrcoef(c_single[valid], c_multi[valid])[0, 1]
        return np.nan
    except Exception:
        return np.nan

neurons_df["rf_corr"] = neurons_df.apply(get_rf_corr, axis=1)
neurons_df["rf_ipsi_corr"] = neurons_df.apply(get_rf_ipsi_corr, axis=1)

# Identify depth-tuned neurons
is_depth_tuned = (neurons_df["iscell"] == 1) & (
    depth_selectivity_plots.common_utils.one_sided_pval_from_spearman(
        neurons_df["depth_tuning_test_spearmanr_rval_closedloop"],
        neurons_df["depth_tuning_test_spearmanr_pval_closedloop"],
    )
    < 0.05
)
neurons_df["is_depth_tuned"] = is_depth_tuned
sig_depth_df = neurons_df[(neurons_df["rf_sig"] == 1) & (neurons_df["is_depth_tuned"])].copy()
print(f"Loaded {len(neurons_df)} total neurons, {len(sig_depth_df)} depth-tuned with significant RF.")

In [ ]:
# Load example session data for tuning curves and RF maps
example_session = "PZAH17.1e_S20250318"
neurons_df_example = pd.read_pickle(
    pipeline_utils.create_neurons_ds(
        session_name=example_session,
        flexilims_session=flexilims_session,
        project=None,
        conflicts="skip",
    ).path_full
)
_, trials_df_example = spheres.sync_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)
depths = np.sort(trials_df_example.depth.unique())
print(f"Example session {example_session} loaded with {len(neurons_df_example)} neurons.")

## 2. Layout Configuration & Helper Functions

In [ ]:
FIG_W = 18.0   # cm
FIG_H = 11.55  # cm (increased by 10% from 10.5 cm)
FONTSIZE_DICT = {"title": 7, "label": 7, "tick": 6, "legend": 5.5}

def rect_cm(fig, x, y, w, h):
    """Convert centimetre coordinates (bottom-left origin) to figure fraction bounds."""
    fig_w, fig_h = np.array(fig.get_size_inches()) * 2.54
    return [x / fig_w, y / fig_h, w / fig_w, h / fig_h]

def panel_letter(fig, letter, x, y, fontsize=8):
    """Draw bold panel letter annotation in cm coordinates."""
    fig_w, fig_h = np.array(fig.get_size_inches()) * 2.54
    fig.text(x / fig_w, y / fig_h, letter, fontsize=fontsize, fontweight="bold", va="top", ha="left")


## 3. Panels A–C: Example Neurons (Single-Depth vs. Multi-Depth)

In [ ]:
# Example neurons representing Near, Mid, and Far depth tuning (distinct from main fig ROIs 53, 118, 623)
EXAMPLE_ROIS = [563, 186, 497]

def plot_example_panels(fig, neurons_df, trials_df, rois=EXAMPLE_ROIS):
    col_w = 3.4
    col_gap = 0.5
    rf_h = 7.7  # increased height
    plot_prop = 0.90
    h_slice = rf_h / ndepths * plot_prop
    rf_w = 1.5 * h_slice
    rf_gap = 0.10
    tune_w = 2 * rf_w + rf_gap  # Matched to 2-column matrix width
    
    for i_ex, roi in enumerate(rois):
        x_base = 0.6 + i_ex * (col_w + col_gap)
        
        # Depth tuning curve (black trace & points, large marker size)
        ax_tune = fig.add_axes(rect_cm(fig, x_base + 0.1, 9.7, tune_w, 1.3))
        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_df,
            trials_df=trials_df,
            roi=roi,
            linecolor="k",
            fit_linecolor="k",
            markeredgecolor="k",
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1.2,
            closed_loop=1,
            fontsize_dict=FONTSIZE_DICT,
            markersize=6.5,
            ylim_precision_base=5,
            ylim_precision=2,
        )
        ax_tune.set_ylabel(r"$\Delta F/F_0$" if i_ex == 0 else "", fontsize=FONTSIZE_DICT["label"])
        ax_tune.set_xlabel("Virtual depth (cm)", fontsize=FONTSIZE_DICT["label"], labelpad=1)
        xticks = ax_tune.get_xticks()
        xlabels = [t.get_text() for t in ax_tune.get_xticklabels()]
        ax_tune.set_xticks(xticks[::2], xlabels[::2], rotation=45, fontsize=FONTSIZE_DICT["tick"])
        ax_tune.tick_params(length=1.5, pad=1)
        
        # Compute shared clim for consistent scaling across single and multi depth
        c_s = neurons_df.loc[roi, "rf_coef_closedloop"][:, :-1]
        c_m = neurons_df.loc[roi, "rf_coef_closedloop_multidepth"][:, :-1]
        c_s_mean = np.nanmean(c_s.reshape(c_s.shape[0], ndepths, 16, 24), axis=0)
        c_m_mean = np.nanmean(c_m.reshape(c_m.shape[0], ndepths, 16, 24), axis=0)
        clim = max(np.nanmax(np.abs(c_s_mean)), np.nanmax(np.abs(c_m_mean)), 0.1)
        
        # Extended Single depth RF stack (8 depths) - no colorbar, title on top axis
        axes_s = rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=rect_cm(fig, x_base + 0.1, 7.6, rf_w, rf_h),
            plot_prop=plot_prop,
            xlabel="",
            ylabel="Elevation (°)" if i_ex == 0 else "",
            fontsize_dict=FONTSIZE_DICT,
            use_multidepth=False,
            clim=clim,
            plot_yticklabels=(i_ex == 0),
            colorbar=False,
        )
        axes_s[0].set_title("Single depth", fontsize=FONTSIZE_DICT["label"], pad=2)
        
        # Extended Multi depth RF stack (8 depths) - tight next to single depth, keep shared colorbar
        axes_m = rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=rect_cm(fig, x_base + 0.1 + rf_w + rf_gap, 7.6, rf_w, rf_h),
            plot_prop=plot_prop,
            xlabel="",
            ylabel="",
            fontsize_dict=FONTSIZE_DICT,
            use_multidepth=True,
            clim=clim,
            plot_yticklabels=False,
            colorbar=True,
        )
        axes_m[0].set_title("Multi depth", fontsize=FONTSIZE_DICT["label"], pad=2)
        
        # Single centered Azimuth label per panel (A, B, C)
        x_center_rf = x_base + 0.1 + rf_w + rf_gap / 2
        fig.text(
            x_center_rf / FIG_W,
            0.28 / FIG_H,
            "Azimuth (°)",
            ha="center",
            va="bottom",
            fontsize=FONTSIZE_DICT["label"],
        )


## 4. Panels D–E: Population Similarity Metrics


In [ ]:
def plot_population_panels(fig, df):
    # Only include neurons with significant RF
    df_rf = df[df["rf_sig"] == 1].copy() if "rf_sig" in df.columns else df.copy()

    # Panel D: Top of right third (RF Correlation Histogram vs Ipsi Control)
    ax_d = fig.add_axes(rect_cm(fig, 13.2, 6.7, 4.2, 3.6))
    corr_contra = df_rf["rf_corr"].dropna()
    corr_ipsi = df_rf["rf_ipsi_corr"].dropna()
    
    bins = np.linspace(-0.4, 1.0, 25)
    ax_d.hist(corr_contra, bins=bins, density=True, alpha=0.6, color="royalblue", label="Contralateral", edgecolor="none")
    ax_d.hist(corr_ipsi, bins=bins, density=True, alpha=0.5, color="gray", label="Ipsilateral", edgecolor="none")
    ax_d.axvline(corr_contra.median(), color="blue", linestyle="--", linewidth=1.2)
    ax_d.axvline(corr_ipsi.median(), color="dimgray", linestyle="--", linewidth=1.2)
    ax_d.set_xlabel("RF correlation (Pearson r)", fontsize=FONTSIZE_DICT["label"])
    ax_d.set_ylabel("Density", fontsize=FONTSIZE_DICT["label"])
    ax_d.legend(
        fontsize=FONTSIZE_DICT["legend"],
        frameon=False,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.02),
        ncol=2,
        handletextpad=0.4,
        columnspacing=0.8,
    )
    ax_d.set_xlim(-0.4, 1.0)
    ax_d.set_xticks([-0.4, 0.0, 0.5, 1.0])
    ax_d.set_yticks([0, 1, 2, 3, 4])
    ax_d.tick_params(labelsize=FONTSIZE_DICT["tick"], length=1.5)
    sns.despine(ax=ax_d, trim=True)
    
    # Panel E: Bottom of right third (Single-depth RF peak vs Multi-depth RF peak in cm)
    ax_e = fig.add_axes(rect_cm(fig, 13.2, 1.0, 4.0, 4.0))
    df_depth = df_rf[df_rf["is_depth_tuned"]] if "is_depth_tuned" in df_rf.columns else df_rf
    valid_pref_d = df_depth[["rf_preferred_depth_closedloop", "rf_preferred_depth_closedloop_multidepth"]].dropna()
    
    x_cm = valid_pref_d["rf_preferred_depth_closedloop"] * 100
    y_cm = valid_pref_d["rf_preferred_depth_closedloop_multidepth"] * 100
    
    xlim = [1.5, 2500]
    ylim = [4.5, 2500]
    ax_e.scatter(x_cm, y_cm, s=8, alpha=0.35, color="darkorchid", edgecolors="none", clip_on=False)
    ax_e.set_xscale("log")
    ax_e.set_yscale("log")
    ax_e.plot([ylim[0], xlim[1]], [ylim[0], xlim[1]], "k--", linewidth=0.8, alpha=0.7)
    ax_e.set_xlim(xlim)
    ax_e.set_ylim(ylim)
    
    # Explicit scalar centimetre tick labels (10, 100, 1000 cm instead of 10^1, 10^2, 10^3)
    ax_e.set_xticks([10, 100, 1000])
    ax_e.set_xticklabels(["10", "100", "1000"])
    ax_e.set_yticks([10, 100, 1000])
    ax_e.set_yticklabels(["10", "100", "1000"])
    
    ax_e.set_aspect("equal")
    ax_e.tick_params(which="major", length=3.0, labelsize=FONTSIZE_DICT["tick"])
    ax_e.tick_params(which="minor", length=1.5)
    ax_e.set_xlabel("Single-depth RF peak depth (cm)", fontsize=FONTSIZE_DICT["label"])
    ax_e.set_ylabel("Multi-depth RF peak depth (cm)", fontsize=FONTSIZE_DICT["label"])
    sns.despine(ax=ax_e)
    ax_e.spines["bottom"].set_bounds(xlim[0], xlim[1])
    ax_e.spines["left"].set_bounds(ylim[0], ylim[1])


## 5. Assemble and Save Unified Publication Figure

In [ ]:
# Assemble unified publication canvas
fig = plt.figure(figsize=(FIG_W / 2.54, FIG_H / 2.54), dpi=300)
bg_ax = fig.add_axes([0, 0, 1, 1])
bg_ax.set_xticks([])
bg_ax.set_yticks([])
fig.patch.set_facecolor("white")

# Panel letters
panel_letter(fig, "A", 0.1, 11.2)
panel_letter(fig, "B", 3.6, 11.2)
panel_letter(fig, "C", 7.5, 11.2)
panel_letter(fig, "D", 12.3, 11.2)
panel_letter(fig, "E", 12.3, 5.3)

# Draw example neuron panels A, B, C (left 2/3)
plot_example_panels(fig, neurons_df_example, trials_df_example)

# Draw population panels D, E (right 1/3, stacked vertically)
plot_population_panels(fig, neurons_df)

# Export publication-ready vector and raster formats
save_path = SAVE_ROOT / "fig_supp_multidepth_rf" / "figsupp_multidepth_receptive_fields"
for ext in ["pdf", "svg", "png"]:
    fig.savefig(f"{save_path}.{ext}", dpi=300, bbox_inches="tight")
print(f"Supplementary Figure successfully generated and saved to:\n  {save_path}.png/.pdf/.svg")
plt.show()


## 6. Spatial Distance Between Single-Depth and Multi-Depth RF Peaks

Calculates and plots the distribution of Euclidean angular distance (in degrees) on the azimuth/elevation plane between the receptive field peak under single-depth vs. multi-depth stimulus conditions.

In [ ]:
# 1. Compute RF centers on the Az/El plane for Single depth
coef_single = np.stack(neurons_df["rf_coef_closedloop"].values)
azi_s, ele_s, _, _ = rf_analysis.find_rf_centers(
    neurons_df=neurons_df,
    ndepths=ndepths,
    frame_shape=(16, 24),
    is_closed_loop=1,
    resolution=5,
    use_multidepth=False,
    coef=coef_single,
)

# 2. Compute RF centers on the Az/El plane for Multi depth
coef_multi = np.stack(neurons_df["rf_coef_closedloop_multidepth"].values)
azi_m, ele_m, _, _ = rf_analysis.find_rf_centers(
    neurons_df=neurons_df,
    ndepths=ndepths,
    frame_shape=(16, 24),
    is_closed_loop=1,
    resolution=5,
    use_multidepth=True,
    coef=coef_multi,
)

# 3. Compute Euclidean distance in degrees on the Azimuth/Elevation plane
neurons_df["rf_azi_single"] = azi_s
neurons_df["rf_ele_single"] = ele_s
neurons_df["rf_azi_multi"] = azi_m
neurons_df["rf_ele_multi"] = ele_m
neurons_df["rf_dist_az_el_deg"] = np.sqrt((azi_m - azi_s) ** 2 + (ele_m - ele_s) ** 2)

# 4. Filter for neurons with significant RF
df_rf = neurons_df[neurons_df["rf_sig"] == 1].copy()
valid_dists = df_rf["rf_dist_az_el_deg"].dropna()

# 5. Plot distribution of RF peak distances
fig_dist, ax_dist = plt.subplots(figsize=(6.5 / 2.54, 5.5 / 2.54), dpi=300)
bins = np.arange(0, 85, 5)  # 5° bins matching monitor spatial grid resolution
ax_dist.hist(
    valid_dists,
    bins=bins,
    density=True,
    color="teal",
    alpha=0.65,
    edgecolor="white",
    linewidth=0.5,
)

med_dist = valid_dists.median()
q25, q75 = np.percentile(valid_dists, [25, 75])
ax_dist.axvline(
    med_dist,
    color="darkslategrey",
    linestyle="--",
    linewidth=1.2,
    label=f"Median: {med_dist:.1f}° (IQR: {q25:.1f}–{q75:.1f}°)",
)

ax_dist.set_xlabel("RF peak distance in Az/El plane (°)", fontsize=FONTSIZE_DICT["label"])
ax_dist.set_ylabel("Probability density", fontsize=FONTSIZE_DICT["label"])
ax_dist.set_xlim(0, 80)
ax_dist.set_xticks([0, 20, 40, 60, 80])
ax_dist.tick_params(labelsize=FONTSIZE_DICT["tick"], length=2)
ax_dist.legend(fontsize=FONTSIZE_DICT["legend"], frameon=False, loc="upper right")
sns.despine(ax=ax_dist, trim=True)

# Save supplementary plot
dist_save_path = SAVE_ROOT / "fig_supp_multidepth_rf" / "rf_peak_az_el_distance_distribution.png"
fig_dist.savefig(dist_save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Summary (N = {len(valid_dists)} neurons with significant RF):")
print(f"  Median angular distance: {med_dist:.2f}°")
print(f"  Interquartile range (IQR): [{q25:.2f}°, {q75:.2f}°]")
print(f"  Mean ± std: {valid_dists.mean():.2f}° ± {valid_dists.std():.2f}°")
